### New Addition

In [0]:
# Configures Gold table names, runtime parameters, evaluation settings, and pipeline metadata.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NAMESPACE = f"{CATALOG}.{SCHEMA}"

SILVER_PRODUCTS_TABLE = f"{NAMESPACE}.silver_product_limits"
SILVER_SHIPMENTS_TABLE = f"{NAMESPACE}.silver_shipments"
SILVER_DELIVERIES_TABLE = f"{NAMESPACE}.silver_delivery_events"
SILVER_SENSOR_TABLE = f"{NAMESPACE}.silver_sensor_events"
SILVER_SENSOR_ENRICHED_TABLE = f"{NAMESPACE}.silver_sensor_enriched"
QUARANTINE_SENSOR_TABLE = f"{NAMESPACE}.quarantine_sensor_events"

TEMPERATURE_WINDOWS_TABLE = f"{NAMESPACE}.shipment_temperature_windows"
SHIPMENT_HEALTH_TABLE = f"{NAMESPACE}.shipment_health"
DELIVERY_PERFORMANCE_TABLE = f"{NAMESPACE}.delivery_performance"

dbutils.widgets.text(
    "run_id",
    ""
)

dbutils.widgets.text(
    "as_of_time",
    ""
)

dbutils.widgets.text(
    "temperature_window",
    "5 minutes"
)

dbutils.widgets.text(
    "stale_threshold_minutes",
    "10"
)

RUN_ID_INPUT = dbutils.widgets.get(
    "run_id"
).strip()

AS_OF_TIME_INPUT = dbutils.widgets.get(
    "as_of_time"
).strip()

TEMPERATURE_WINDOW = dbutils.widgets.get(
    "temperature_window"
).strip()

STALE_THRESHOLD_MINUTES = int(
    dbutils.widgets.get(
        "stale_threshold_minutes"
    )
)

RUN_ID = (
    RUN_ID_INPUT
    if RUN_ID_INPUT
    else str(uuid.uuid4())
)

assert STALE_THRESHOLD_MINUTES > 0, (
    "stale_threshold_minutes must be greater than zero"
)

print("==============================================")
print("GOLD PIPELINE CONFIGURATION")
print("==============================================")

print("Namespace               :", NAMESPACE)
print("Run ID                  :", RUN_ID)
print("As-of-time input        :", AS_OF_TIME_INPUT)
print("Temperature window      :", TEMPERATURE_WINDOW)
print("Stale threshold minutes :", STALE_THRESHOLD_MINUTES)

In [0]:
# Verifies required Silver tables and loads the validated datasets used by the Gold transformations.

required_source_tables = [
    SILVER_PRODUCTS_TABLE,
    SILVER_SHIPMENTS_TABLE,
    SILVER_DELIVERIES_TABLE,
    SILVER_SENSOR_TABLE,
    SILVER_SENSOR_ENRICHED_TABLE,
    QUARANTINE_SENSOR_TABLE
]

missing_source_tables = [
    table_name
    for table_name in required_source_tables
    if not spark.catalog.tableExists(
        table_name
    )
]

assert len(missing_source_tables) == 0, (
    "Missing required source tables: "
    + ", ".join(
        missing_source_tables
    )
)

products = spark.table(
    SILVER_PRODUCTS_TABLE
)

shipments = spark.table(
    SILVER_SHIPMENTS_TABLE
)

deliveries = spark.table(
    SILVER_DELIVERIES_TABLE
)

sensors = spark.table(
    SILVER_SENSOR_TABLE
)

enriched_sensors = spark.table(
    SILVER_SENSOR_ENRICHED_TABLE
)

quarantine_sensors = spark.table(
    QUARANTINE_SENSOR_TABLE
)

required_shipment_columns = {
    "shipment_id",
    "product_id",
    "vehicle_id",
    "origin",
    "destination",
    "dispatch_time",
    "expected_arrival"
}

required_delivery_columns = {
    "event_id",
    "shipment_id",
    "status",
    "event_time"
}

required_enriched_sensor_columns = {
    "event_id",
    "shipment_id",
    "sensor_id",
    "event_time",
    "temperature_c",
    "product_id",
    "min_temperature",
    "max_temperature",
    "is_temperature_exception"
}

missing_shipment_columns = (
    required_shipment_columns
    - set(shipments.columns)
)

missing_delivery_columns = (
    required_delivery_columns
    - set(deliveries.columns)
)

missing_enriched_sensor_columns = (
    required_enriched_sensor_columns
    - set(enriched_sensors.columns)
)

assert len(missing_shipment_columns) == 0, (
    "Missing Silver shipment columns: "
    + ", ".join(
        sorted(missing_shipment_columns)
    )
)

assert len(missing_delivery_columns) == 0, (
    "Missing Silver delivery columns: "
    + ", ".join(
        sorted(missing_delivery_columns)
    )
)

assert len(missing_enriched_sensor_columns) == 0, (
    "Missing enriched sensor columns: "
    + ", ".join(
        sorted(missing_enriched_sensor_columns)
    )
)

if AS_OF_TIME_INPUT:
    evaluation_time = (
        spark.range(1)
        .select(
            F.to_timestamp(
                F.lit(AS_OF_TIME_INPUT)
            ).alias("evaluation_time")
        )
        .first()["evaluation_time"]
    )

    assert evaluation_time is not None, (
        f"Invalid as_of_time: {AS_OF_TIME_INPUT}"
    )
else:
    evaluation_time = (
        shipments
        .agg(
            (
                F.max("expected_arrival")
                + F.expr("INTERVAL 1 HOUR")
            ).alias("evaluation_time")
        )
        .first()["evaluation_time"]
    )

assert evaluation_time is not None, (
    "Could not determine the Gold evaluation time"
)

print("==============================================")
print("GOLD SOURCE TABLES")
print("==============================================")

print("Silver products          :", products.count())
print("Silver shipments         :", shipments.count())
print("Silver deliveries        :", deliveries.count())
print("Silver sensors           :", sensors.count())
print("Enriched sensors         :", enriched_sensors.count())
print("Quarantined sensors      :", quarantine_sensors.count())
print("Evaluation time          :", evaluation_time)

In [0]:
# Aggregates enriched sensor readings into configurable event-time windows for each shipment.

temperature_windows = (
    enriched_sensors
    .groupBy(
        F.col("shipment_id"),
        F.window(
            F.col("event_time"),
            TEMPERATURE_WINDOW
        ).alias("event_window")
    )
    .agg(
        F.min(
            "temperature_c"
        ).alias(
            "minimum_temperature_c"
        ),
        F.max(
            "temperature_c"
        ).alias(
            "maximum_temperature_c"
        ),
        F.avg(
            "temperature_c"
        ).alias(
            "average_temperature_c"
        ),
        F.count(
            F.lit(1)
        ).alias(
            "accepted_reading_count"
        ),
        F.sum(
            F.when(
                F.col(
                    "is_temperature_exception"
                ) == True,
                F.lit(1)
            ).otherwise(
                F.lit(0)
            )
        ).cast("long").alias(
            "exception_count"
        ),
        F.min(
            "event_time"
        ).alias(
            "first_event_time"
        ),
        F.max(
            "event_time"
        ).alias(
            "last_event_time"
        )
    )
    .select(
        F.col("shipment_id"),
        F.col(
            "event_window.start"
        ).alias(
            "window_start"
        ),
        F.col(
            "event_window.end"
        ).alias(
            "window_end"
        ),
        F.col("first_event_time"),
        F.col("last_event_time"),
        F.col("minimum_temperature_c"),
        F.col("maximum_temperature_c"),
        F.col("average_temperature_c"),
        F.col("accepted_reading_count"),
        F.col("exception_count")
    )
    .withColumn(
        "normal_reading_count",
        F.col("accepted_reading_count")
        - F.col("exception_count")
    )
    .withColumn(
        "exception_rate",
        (
            F.col("exception_count").cast("double")
            / F.col(
                "accepted_reading_count"
            ).cast("double")
        ) * F.lit(100.0)
    )
    .withColumn(
        "has_temperature_exception",
        F.col("exception_count") > 0
    )
    .withColumn(
        "evaluated_at",
        F.lit(
            evaluation_time
        ).cast("timestamp")
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
)

temperature_window_count = (
    temperature_windows.count()
)

gold_window_reading_count = (
    temperature_windows
    .agg(
        F.sum(
            "accepted_reading_count"
        ).alias(
            "total_readings"
        )
    )
    .first()["total_readings"]
)

silver_enriched_count = (
    enriched_sensors.count()
)

assert gold_window_reading_count == silver_enriched_count, (
    f"Temperature-window reconciliation failed: "
    f"Gold={gold_window_reading_count}, "
    f"Silver={silver_enriched_count}"
)

(
    temperature_windows.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TEMPERATURE_WINDOWS_TABLE
    )
)

print("==============================================")
print("SHIPMENT TEMPERATURE WINDOWS")
print("==============================================")

print(
    "Gold window rows       :",
    temperature_window_count
)

print(
    "Gold represented reads :",
    gold_window_reading_count
)

print(
    "Silver enriched reads  :",
    silver_enriched_count
)

print(
    "Written table          :",
    TEMPERATURE_WINDOWS_TABLE
)

display(
    spark.table(
        TEMPERATURE_WINDOWS_TABLE
    )
    .orderBy(
        "shipment_id",
        "window_start"
    )
)

In [0]:
# Validates Gold temperature windows for count reconciliation, ranges, boundaries, and exception classification.

saved_temperature_windows = spark.table(
    TEMPERATURE_WINDOWS_TABLE
)

saved_enriched_sensors = spark.table(
    SILVER_SENSOR_ENRICHED_TABLE
)

window_count = (
    saved_temperature_windows.count()
)

gold_accepted_readings = (
    saved_temperature_windows
    .agg(
        F.sum(
            "accepted_reading_count"
        ).alias(
            "total"
        )
    )
    .first()["total"]
)

gold_exception_count = (
    saved_temperature_windows
    .agg(
        F.sum(
            "exception_count"
        ).alias(
            "total"
        )
    )
    .first()["total"]
)

silver_reading_count = (
    saved_enriched_sensors.count()
)

silver_exception_count = (
    saved_enriched_sensors
    .filter(
        F.col(
            "is_temperature_exception"
        ) == True
    )
    .count()
)

invalid_window_boundaries = (
    saved_temperature_windows
    .filter(
        F.col("window_end")
        <= F.col("window_start")
    )
    .count()
)

invalid_window_counts = (
    saved_temperature_windows
    .filter(
        (F.col("accepted_reading_count") <= 0)
        | (F.col("exception_count") < 0)
        | (F.col("normal_reading_count") < 0)
        | (
            F.col("exception_count")
            > F.col("accepted_reading_count")
        )
        | (
            F.col("normal_reading_count")
            + F.col("exception_count")
            != F.col("accepted_reading_count")
        )
    )
    .count()
)

invalid_exception_rates = (
    saved_temperature_windows
    .filter(
        F.col("exception_rate").isNull()
        | (F.col("exception_rate") < 0)
        | (F.col("exception_rate") > 100)
    )
    .count()
)

invalid_temperature_ranges = (
    saved_temperature_windows
    .filter(
        F.col("minimum_temperature_c")
        > F.col("maximum_temperature_c")
    )
    .count()
)

invalid_exception_flags = (
    saved_temperature_windows
    .filter(
        (
            F.col("exception_count") > 0
        )
        != F.col(
            "has_temperature_exception"
        )
    )
    .count()
)

assert gold_accepted_readings == silver_reading_count, (
    f"Accepted-reading reconciliation failed: "
    f"Gold={gold_accepted_readings}, "
    f"Silver={silver_reading_count}"
)

assert gold_exception_count == silver_exception_count, (
    f"Exception reconciliation failed: "
    f"Gold={gold_exception_count}, "
    f"Silver={silver_exception_count}"
)

assert invalid_window_boundaries == 0, (
    f"Found {invalid_window_boundaries} "
    "invalid window boundaries"
)

assert invalid_window_counts == 0, (
    f"Found {invalid_window_counts} "
    "invalid window count rows"
)

assert invalid_exception_rates == 0, (
    f"Found {invalid_exception_rates} "
    "invalid exception rates"
)

assert invalid_temperature_ranges == 0, (
    f"Found {invalid_temperature_ranges} "
    "invalid temperature ranges"
)

assert invalid_exception_flags == 0, (
    f"Found {invalid_exception_flags} "
    "incorrect exception flags"
)

print("==============================================")
print("TEMPERATURE WINDOW VALIDATION")
print("==============================================")

print("Gold window rows             :", window_count)
print("Gold accepted readings       :", gold_accepted_readings)
print("Silver enriched readings     :", silver_reading_count)
print("Gold temperature exceptions  :", gold_exception_count)
print("Silver temperature exceptions:", silver_exception_count)
print("Invalid boundaries           :", invalid_window_boundaries)
print("Invalid counts               :", invalid_window_counts)
print("Invalid exception rates      :", invalid_exception_rates)
print("Invalid temperature ranges   :", invalid_temperature_ranges)
print("Invalid exception flags      :", invalid_exception_flags)

print()
print("Temperature window validation passed.")

In [0]:
# Builds one operational-health record per shipment using latest sensors, delivery state, and stale-reading logic.

latest_sensor_window = (
    Window
    .partitionBy(
        "shipment_id"
    )
    .orderBy(
        F.col("event_time").desc(),
        F.col("event_id").desc()
    )
)

latest_sensor = (
    enriched_sensors
    .withColumn(
        "_latest_sensor_rank",
        F.row_number().over(
            latest_sensor_window
        )
    )
    .filter(
        F.col("_latest_sensor_rank") == 1
    )
    .select(
        F.col("shipment_id"),
        F.col("event_id").alias(
            "latest_event_id"
        ),
        F.col("sensor_id").alias(
            "latest_sensor_id"
        ),
        F.col("event_time").alias(
            "latest_event_time"
        ),
        F.col("temperature_c").alias(
            "latest_temperature_c"
        ),
        F.col(
            "is_temperature_exception"
        ).alias(
            "latest_is_temperature_exception"
        )
    )
)

latest_delivery_window = (
    Window
    .partitionBy(
        "shipment_id"
    )
    .orderBy(
        F.col("event_time").desc(),
        F.col("event_id").desc()
    )
)

latest_delivery = (
    deliveries
    .filter(
        F.upper(
            F.trim(
                F.col("status")
            )
        ) == "DELIVERED"
    )
    .withColumn(
        "_latest_delivery_rank",
        F.row_number().over(
            latest_delivery_window
        )
    )
    .filter(
        F.col("_latest_delivery_rank") == 1
    )
    .select(
        F.col("shipment_id"),
        F.col("event_time").alias(
            "actual_arrival"
        )
    )
)

shipment_health = (
    shipments.alias("shipment")
    .join(
        latest_sensor.alias("sensor"),
        on="shipment_id",
        how="left"
    )
    .join(
        latest_delivery.alias("delivery"),
        on="shipment_id",
        how="left"
    )
    .withColumn(
        "evaluated_at",
        F.lit(
            evaluation_time
        ).cast("timestamp")
    )
    .withColumn(
        "shipment_status",
        F.when(
            F.col("actual_arrival").isNotNull(),
            F.lit("DELIVERED")
        )
        .when(
            F.col("evaluated_at")
            > F.col("expected_arrival"),
            F.lit("ACTIVE_OVERDUE")
        )
        .otherwise(
            F.lit("ACTIVE_IN_TRANSIT")
        )
    )
    .withColumn(
        "current_temperature_status",
        F.when(
            F.col("latest_event_time").isNull(),
            F.lit("NO_ACCEPTED_READING")
        )
        .when(
            F.col(
                "latest_is_temperature_exception"
            ) == True,
            F.lit("TEMPERATURE_EXCEPTION")
        )
        .otherwise(
            F.lit("NORMAL")
        )
    )
    .withColumn(
        "sensor_age_minutes",
        F.when(
            F.col("latest_event_time").isNotNull(),
            (
                F.unix_timestamp(
                    "evaluated_at"
                )
                - F.unix_timestamp(
                    "latest_event_time"
                )
            ) / F.lit(60.0)
        ).otherwise(
            F.lit(None).cast("double")
        )
    )
    .withColumn(
        "sensor_freshness_status",
        F.when(
            F.col("actual_arrival").isNotNull(),
            F.lit("NOT_APPLICABLE")
        )
        .when(
            F.col("latest_event_time").isNull(),
            F.lit("MISSING")
        )
        .when(
            F.col("sensor_age_minutes")
            > F.lit(
                float(
                    STALE_THRESHOLD_MINUTES
                )
            ),
            F.lit("STALE")
        )
        .otherwise(
            F.lit("RECENT")
        )
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .select(
        "shipment_id",
        "product_id",
        "vehicle_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival",
        "actual_arrival",
        "latest_event_id",
        "latest_sensor_id",
        "latest_temperature_c",
        "latest_event_time",
        "sensor_age_minutes",
        "current_temperature_status",
        "shipment_status",
        "sensor_freshness_status",
        "evaluated_at",
        "run_id"
    )
)

(
    shipment_health.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        SHIPMENT_HEALTH_TABLE
    )
)

print("==============================================")
print("SHIPMENT HEALTH")
print("==============================================")

print(
    "Silver shipments :",
    shipments.count()
)

print(
    "Health rows      :",
    spark.table(
        SHIPMENT_HEALTH_TABLE
    ).count()
)

print(
    "Written table    :",
    SHIPMENT_HEALTH_TABLE
)

display(
    spark.table(
        SHIPMENT_HEALTH_TABLE
    )
    .orderBy(
        "shipment_id"
    )
)

In [0]:
# Validates one-row-per-shipment health coverage and all operational status combinations.

saved_shipments = spark.table(
    SILVER_SHIPMENTS_TABLE
)

saved_health = spark.table(
    SHIPMENT_HEALTH_TABLE
)

shipment_count = (
    saved_shipments.count()
)

health_count = (
    saved_health.count()
)

duplicate_health_shipments = (
    saved_health
    .groupBy(
        "shipment_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

missing_health_shipments = (
    saved_shipments.alias("shipment")
    .join(
        saved_health.alias("health"),
        on="shipment_id",
        how="left_anti"
    )
    .count()
)

invalid_shipment_status = (
    saved_health
    .filter(
        ~F.col(
            "shipment_status"
        ).isin(
            "DELIVERED",
            "ACTIVE_OVERDUE",
            "ACTIVE_IN_TRANSIT"
        )
    )
    .count()
)

invalid_temperature_status = (
    saved_health
    .filter(
        ~F.col(
            "current_temperature_status"
        ).isin(
            "NORMAL",
            "TEMPERATURE_EXCEPTION",
            "NO_ACCEPTED_READING"
        )
    )
    .count()
)

invalid_freshness_status = (
    saved_health
    .filter(
        ~F.col(
            "sensor_freshness_status"
        ).isin(
            "MISSING",
            "STALE",
            "RECENT",
            "NOT_APPLICABLE"
        )
    )
    .count()
)

delivered_without_arrival = (
    saved_health
    .filter(
        (
            F.col("shipment_status")
            == "DELIVERED"
        )
        & F.col(
            "actual_arrival"
        ).isNull()
    )
    .count()
)

active_with_arrival = (
    saved_health
    .filter(
        F.col(
            "shipment_status"
        ).startswith(
            "ACTIVE"
        )
        & F.col(
            "actual_arrival"
        ).isNotNull()
    )
    .count()
)

active_not_applicable = (
    saved_health
    .filter(
        F.col(
            "shipment_status"
        ).startswith(
            "ACTIVE"
        )
        & (
            F.col(
                "sensor_freshness_status"
            ) == "NOT_APPLICABLE"
        )
    )
    .count()
)

delivered_sensor_applicable = (
    saved_health
    .filter(
        (
            F.col("shipment_status")
            == "DELIVERED"
        )
        & (
            F.col(
                "sensor_freshness_status"
            ) != "NOT_APPLICABLE"
        )
    )
    .count()
)

assert health_count == shipment_count, (
    f"Expected {shipment_count} health rows, "
    f"found {health_count}"
)

assert duplicate_health_shipments == 0, (
    "Duplicate shipment IDs found in shipment_health"
)

assert missing_health_shipments == 0, (
    f"Found {missing_health_shipments} "
    "shipments without health records"
)

assert invalid_shipment_status == 0, (
    f"Found {invalid_shipment_status} "
    "invalid shipment statuses"
)

assert invalid_temperature_status == 0, (
    f"Found {invalid_temperature_status} "
    "invalid temperature statuses"
)

assert invalid_freshness_status == 0, (
    f"Found {invalid_freshness_status} "
    "invalid freshness statuses"
)

assert delivered_without_arrival == 0, (
    f"Found {delivered_without_arrival} "
    "delivered shipments without arrival times"
)

assert active_with_arrival == 0, (
    f"Found {active_with_arrival} "
    "active shipments with arrival times"
)

assert active_not_applicable == 0, (
    f"Found {active_not_applicable} "
    "active shipments with NOT_APPLICABLE freshness"
)

assert delivered_sensor_applicable == 0, (
    f"Found {delivered_sensor_applicable} "
    "delivered shipments with active freshness status"
)

print("==============================================")
print("SHIPMENT HEALTH VALIDATION")
print("==============================================")

print("Silver shipments                :", shipment_count)
print("Shipment health rows            :", health_count)
print("Duplicate shipment rows         :", duplicate_health_shipments)
print("Missing health rows             :", missing_health_shipments)
print("Invalid shipment statuses       :", invalid_shipment_status)
print("Invalid temperature statuses    :", invalid_temperature_status)
print("Invalid freshness statuses      :", invalid_freshness_status)
print("Delivered without arrival       :", delivered_without_arrival)
print("Active with arrival             :", active_with_arrival)
print("Active with N/A freshness       :", active_not_applicable)

print()
print("Shipment health validation passed.")

display(
    saved_health
    .groupBy(
        "shipment_status",
        "sensor_freshness_status",
        "current_temperature_status"
    )
    .count()
    .orderBy(
        "shipment_status",
        "sensor_freshness_status",
        "current_temperature_status"
    )
)

In [0]:
# Builds one delivery-performance record per shipment using the latest valid delivery event.

latest_delivery_window = (
    Window
    .partitionBy(
        "shipment_id"
    )
    .orderBy(
        F.col("event_time").desc(),
        F.col("event_id").desc()
    )
)

latest_deliveries = (
    deliveries
    .filter(
        F.upper(
            F.trim(
                F.col("status")
            )
        ) == "DELIVERED"
    )
    .withColumn(
        "_delivery_rank",
        F.row_number().over(
            latest_delivery_window
        )
    )
    .filter(
        F.col("_delivery_rank") == 1
    )
    .select(
        F.col("shipment_id"),
        F.col("event_time").alias(
            "actual_arrival"
        )
    )
)

delivery_performance = (
    shipments.alias("shipment")
    .join(
        latest_deliveries.alias("delivery"),
        on="shipment_id",
        how="left"
    )
    .withColumn(
        "evaluated_at",
        F.lit(
            evaluation_time
        ).cast("timestamp")
    )
    .withColumn(
        "delivery_status",
        F.when(
            F.col("actual_arrival").isNull(),
            F.lit("UNDELIVERED")
        )
        .when(
            F.col("actual_arrival")
            < F.col("expected_arrival"),
            F.lit("DELIVERED_EARLY")
        )
        .when(
            F.col("actual_arrival")
            == F.col("expected_arrival"),
            F.lit("DELIVERED_ON_TIME")
        )
        .otherwise(
            F.lit("DELIVERED_LATE")
        )
    )
    .withColumn(
        "arrival_variance_minutes",
        F.when(
            F.col("actual_arrival").isNull(),
            F.lit(None).cast("long")
        )
        .otherwise(
            (
                (
                    F.unix_timestamp(
                        "actual_arrival"
                    )
                    - F.unix_timestamp(
                        "expected_arrival"
                    )
                ) / F.lit(60.0)
            ).cast("long")
        )
    )
    .withColumn(
        "delay_minutes",
        F.when(
            F.col("actual_arrival").isNull(),
            F.lit(None).cast("long")
        )
        .otherwise(
            F.greatest(
                F.lit(0).cast("long"),
                F.col(
                    "arrival_variance_minutes"
                )
            )
        )
    )
    .withColumn(
        "early_minutes",
        F.when(
            F.col("actual_arrival").isNull(),
            F.lit(None).cast("long")
        )
        .otherwise(
            F.greatest(
                F.lit(0).cast("long"),
                -F.col(
                    "arrival_variance_minutes"
                )
            )
        )
    )
    .withColumn(
        "open_status",
        F.when(
            F.col("actual_arrival").isNotNull(),
            F.lit("NOT_APPLICABLE")
        )
        .when(
            F.col("evaluated_at")
            > F.col("expected_arrival"),
            F.lit("OPEN_OVERDUE")
        )
        .otherwise(
            F.lit("OPEN_NOT_OVERDUE")
        )
    )
    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
    .select(
        "shipment_id",
        "product_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival",
        "actual_arrival",
        "delivery_status",
        "arrival_variance_minutes",
        "delay_minutes",
        "early_minutes",
        "open_status",
        "evaluated_at",
        "run_id"
    )
)

(
    delivery_performance.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        DELIVERY_PERFORMANCE_TABLE
    )
)

print("==============================================")
print("DELIVERY PERFORMANCE")
print("==============================================")

print(
    "Silver shipments :",
    shipments.count()
)

print(
    "Performance rows :",
    spark.table(
        DELIVERY_PERFORMANCE_TABLE
    ).count()
)

print(
    "Written table    :",
    DELIVERY_PERFORMANCE_TABLE
)

display(
    spark.table(
        DELIVERY_PERFORMANCE_TABLE
    )
    .orderBy(
        "shipment_id"
    )
)

In [0]:
# Validates delivery-performance completeness, status logic, timing calculations, and shipment uniqueness.

saved_delivery_performance = spark.table(
    DELIVERY_PERFORMANCE_TABLE
)

saved_shipments = spark.table(
    SILVER_SHIPMENTS_TABLE
)

shipment_count = (
    saved_shipments.count()
)

performance_count = (
    saved_delivery_performance.count()
)

duplicate_performance_shipments = (
    saved_delivery_performance
    .groupBy(
        "shipment_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

missing_performance_shipments = (
    saved_shipments.alias("shipment")
    .join(
        saved_delivery_performance.alias(
            "performance"
        ),
        on="shipment_id",
        how="left_anti"
    )
    .count()
)

invalid_delivery_statuses = (
    saved_delivery_performance
    .filter(
        ~F.col(
            "delivery_status"
        ).isin(
            "UNDELIVERED",
            "DELIVERED_EARLY",
            "DELIVERED_ON_TIME",
            "DELIVERED_LATE"
        )
    )
    .count()
)

negative_delays = (
    saved_delivery_performance
    .filter(
        F.col("delay_minutes") < 0
    )
    .count()
)

negative_early_minutes = (
    saved_delivery_performance
    .filter(
        F.col("early_minutes") < 0
    )
    .count()
)

undelivered_with_arrival = (
    saved_delivery_performance
    .filter(
        (
            F.col("delivery_status")
            == "UNDELIVERED"
        )
        & F.col(
            "actual_arrival"
        ).isNotNull()
    )
    .count()
)

delivered_without_arrival = (
    saved_delivery_performance
    .filter(
        F.col(
            "delivery_status"
        ).startswith(
            "DELIVERED"
        )
        & F.col(
            "actual_arrival"
        ).isNull()
    )
    .count()
)

incorrect_late_status = (
    saved_delivery_performance
    .filter(
        (
            F.col("delivery_status")
            == "DELIVERED_LATE"
        )
        & (
            F.col("arrival_variance_minutes")
            <= 0
        )
    )
    .count()
)

incorrect_early_status = (
    saved_delivery_performance
    .filter(
        (
            F.col("delivery_status")
            == "DELIVERED_EARLY"
        )
        & (
            F.col("arrival_variance_minutes")
            >= 0
        )
    )
    .count()
)

incorrect_on_time_status = (
    saved_delivery_performance
    .filter(
        (
            F.col("delivery_status")
            == "DELIVERED_ON_TIME"
        )
        & (
            F.col("arrival_variance_minutes")
            != 0
        )
    )
    .count()
)

incorrect_open_status = (
    saved_delivery_performance
    .filter(
        (
            F.col("actual_arrival").isNotNull()
            & (
                F.col("open_status")
                != "NOT_APPLICABLE"
            )
        )
        |
        (
            F.col("actual_arrival").isNull()
            & (
                F.col("evaluated_at")
                > F.col("expected_arrival")
            )
            & (
                F.col("open_status")
                != "OPEN_OVERDUE"
            )
        )
        |
        (
            F.col("actual_arrival").isNull()
            & (
                F.col("evaluated_at")
                <= F.col("expected_arrival")
            )
            & (
                F.col("open_status")
                != "OPEN_NOT_OVERDUE"
            )
        )
    )
    .count()
)

assert performance_count == shipment_count, (
    f"Expected {shipment_count} delivery rows, "
    f"found {performance_count}"
)

assert duplicate_performance_shipments == 0, (
    "Duplicate shipments found in delivery_performance"
)

assert missing_performance_shipments == 0, (
    f"Found {missing_performance_shipments} "
    "shipments without performance rows"
)

assert invalid_delivery_statuses == 0, (
    f"Found {invalid_delivery_statuses} "
    "invalid delivery statuses"
)

assert negative_delays == 0, (
    f"Found {negative_delays} negative delays"
)

assert negative_early_minutes == 0, (
    f"Found {negative_early_minutes} "
    "negative early-minute values"
)

assert undelivered_with_arrival == 0, (
    f"Found {undelivered_with_arrival} "
    "undelivered shipments with arrival times"
)

assert delivered_without_arrival == 0, (
    f"Found {delivered_without_arrival} "
    "delivered shipments without arrival times"
)

assert incorrect_late_status == 0, (
    f"Found {incorrect_late_status} "
    "incorrect late classifications"
)

assert incorrect_early_status == 0, (
    f"Found {incorrect_early_status} "
    "incorrect early classifications"
)

assert incorrect_on_time_status == 0, (
    f"Found {incorrect_on_time_status} "
    "incorrect on-time classifications"
)

assert incorrect_open_status == 0, (
    f"Found {incorrect_open_status} "
    "incorrect open statuses"
)

print("==============================================")
print("DELIVERY PERFORMANCE VALIDATION")
print("==============================================")

print("Silver shipments             :", shipment_count)
print("Delivery performance rows    :", performance_count)
print("Duplicate shipment rows      :", duplicate_performance_shipments)
print("Missing performance rows     :", missing_performance_shipments)
print("Invalid delivery statuses    :", invalid_delivery_statuses)
print("Negative delays              :", negative_delays)
print("Negative early minutes       :", negative_early_minutes)
print("Undelivered with arrival     :", undelivered_with_arrival)
print("Delivered without arrival    :", delivered_without_arrival)
print("Incorrect late statuses      :", incorrect_late_status)
print("Incorrect early statuses     :", incorrect_early_status)
print("Incorrect on-time statuses   :", incorrect_on_time_status)
print("Incorrect open statuses      :", incorrect_open_status)

print()
print("Delivery performance validation passed.")

display(
    saved_delivery_performance
    .groupBy(
        "delivery_status",
        "open_status"
    )
    .count()
    .orderBy(
        "delivery_status",
        "open_status"
    )
)

In [0]:
# Produces the final scale-independent Gold acceptance certificate across all persisted Gold tables.

temperature_windows_final = spark.table(
    TEMPERATURE_WINDOWS_TABLE
)

shipment_health_final = spark.table(
    SHIPMENT_HEALTH_TABLE
)

delivery_performance_final = spark.table(
    DELIVERY_PERFORMANCE_TABLE
)

silver_shipments_final = spark.table(
    SILVER_SHIPMENTS_TABLE
)

silver_enriched_final = spark.table(
    SILVER_SENSOR_ENRICHED_TABLE
)

silver_sensors_final = spark.table(
    SILVER_SENSOR_TABLE
)

quarantine_sensors_final = spark.table(
    QUARANTINE_SENSOR_TABLE
)

bronze_sensors_final = spark.table(
    f"{NAMESPACE}.bronze_sensor_events"
)

bronze_sensor_count = (
    bronze_sensors_final.count()
)

silver_sensor_count = (
    silver_sensors_final.count()
)

quarantine_sensor_count = (
    quarantine_sensors_final.count()
)

silver_enriched_count = (
    silver_enriched_final.count()
)

silver_shipment_count = (
    silver_shipments_final.count()
)

temperature_window_count = (
    temperature_windows_final.count()
)

shipment_health_count = (
    shipment_health_final.count()
)

delivery_performance_count = (
    delivery_performance_final.count()
)

sensor_reconciled_count = (
    silver_sensor_count
    + quarantine_sensor_count
)

gold_window_reading_count = (
    temperature_windows_final
    .agg(
        F.sum(
            "accepted_reading_count"
        ).alias(
            "total"
        )
    )
    .first()["total"]
)

gold_window_exception_count = (
    temperature_windows_final
    .agg(
        F.sum(
            "exception_count"
        ).alias(
            "total"
        )
    )
    .first()["total"]
)

silver_exception_count = (
    silver_enriched_final
    .filter(
        F.col(
            "is_temperature_exception"
        ) == True
    )
    .count()
)

duplicate_health_shipments = (
    shipment_health_final
    .groupBy(
        "shipment_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_delivery_shipments = (
    delivery_performance_final
    .groupBy(
        "shipment_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

invalid_temperature_rates = (
    temperature_windows_final
    .filter(
        F.col("exception_rate").isNull()
        | (F.col("exception_rate") < 0)
        | (F.col("exception_rate") > 100)
    )
    .count()
)

gold_tables = {
    "shipment_temperature_windows": (
        temperature_windows_final
    ),
    "shipment_health": (
        shipment_health_final
    ),
    "delivery_performance": (
        delivery_performance_final
    )
}

metadata_failures = []

for table_name, dataframe in gold_tables.items():
    required_metadata_columns = {
        "run_id",
        "evaluated_at"
    }

    missing_metadata_columns = (
        required_metadata_columns
        - set(dataframe.columns)
    )

    if missing_metadata_columns:
        metadata_failures.append(
            f"{table_name}: missing columns "
            f"{sorted(missing_metadata_columns)}"
        )

        continue

    missing_run_ids = (
        dataframe
        .filter(
            F.col("run_id").isNull()
            | (F.trim(F.col("run_id")) == "")
        )
        .count()
    )

    missing_evaluation_times = (
        dataframe
        .filter(
            F.col("evaluated_at").isNull()
        )
        .count()
    )

    if missing_run_ids > 0:
        metadata_failures.append(
            f"{table_name}: "
            f"{missing_run_ids} missing run IDs"
        )

    if missing_evaluation_times > 0:
        metadata_failures.append(
            f"{table_name}: "
            f"{missing_evaluation_times} "
            f"missing evaluation times"
        )

assert sensor_reconciled_count == bronze_sensor_count, (
    f"Sensor reconciliation failed: "
    f"Silver={silver_sensor_count}, "
    f"Quarantine={quarantine_sensor_count}, "
    f"Bronze={bronze_sensor_count}"
)

assert silver_enriched_count == silver_sensor_count, (
    f"Silver enrichment reconciliation failed: "
    f"Sensor={silver_sensor_count}, "
    f"Enriched={silver_enriched_count}"
)

assert gold_window_reading_count == silver_enriched_count, (
    f"Temperature-window reconciliation failed: "
    f"Gold={gold_window_reading_count}, "
    f"Silver={silver_enriched_count}"
)

assert gold_window_exception_count == silver_exception_count, (
    f"Temperature-exception reconciliation failed: "
    f"Gold={gold_window_exception_count}, "
    f"Silver={silver_exception_count}"
)

assert shipment_health_count == silver_shipment_count, (
    f"Shipment-health reconciliation failed: "
    f"Health={shipment_health_count}, "
    f"Shipments={silver_shipment_count}"
)

assert delivery_performance_count == silver_shipment_count, (
    f"Delivery-performance reconciliation failed: "
    f"Performance={delivery_performance_count}, "
    f"Shipments={silver_shipment_count}"
)

assert duplicate_health_shipments == 0, (
    f"Shipment health contains "
    f"{duplicate_health_shipments} duplicate shipment IDs"
)

assert duplicate_delivery_shipments == 0, (
    f"Delivery performance contains "
    f"{duplicate_delivery_shipments} duplicate shipment IDs"
)

assert invalid_temperature_rates == 0, (
    f"Temperature windows contain "
    f"{invalid_temperature_rates} invalid exception rates"
)

assert len(metadata_failures) == 0, (
    "Gold metadata validation failed: "
    + "; ".join(metadata_failures)
)

print("================================================")
print("FINAL GOLD ACCEPTANCE CERTIFICATE")
print("================================================")

print()
print("SENSOR RECONCILIATION")
print("------------------------------------------------")
print("Bronze sensor rows             :", bronze_sensor_count)
print("Silver sensor rows             :", silver_sensor_count)
print("Quarantine sensor rows         :", quarantine_sensor_count)
print("Reconciled sensor rows         :", sensor_reconciled_count)
print("Enriched sensor rows           :", silver_enriched_count)

print()
print("TEMPERATURE WINDOWS")
print("------------------------------------------------")
print("Temperature window rows        :", temperature_window_count)
print("Gold accepted readings         :", gold_window_reading_count)
print("Gold exception readings        :", gold_window_exception_count)
print("Silver exception readings      :", silver_exception_count)
print("Invalid exception rates        :", invalid_temperature_rates)

print()
print("SHIPMENT GOLD TABLES")
print("------------------------------------------------")
print("Silver shipment rows           :", silver_shipment_count)
print("Shipment health rows           :", shipment_health_count)
print("Delivery performance rows      :", delivery_performance_count)
print("Duplicate health rows          :", duplicate_health_shipments)
print("Duplicate delivery rows        :", duplicate_delivery_shipments)

print()
print("PIPELINE METADATA")
print("------------------------------------------------")
print("Run ID                         :", RUN_ID)
print("Evaluation time                :", evaluation_time)
print("Metadata failures              :", len(metadata_failures))

print()
print("GOLD TABLE DETAILS")
print("------------------------------------------------")

for table_name, dataframe in gold_tables.items():
    table_row_count = dataframe.count()

    table_run_ids = [
        row["run_id"]
        for row in (
            dataframe
            .select(
                "run_id"
            )
            .distinct()
            .orderBy(
                "run_id"
            )
            .collect()
        )
    ]

    earliest_evaluation = (
        dataframe
        .agg(
            F.min(
                "evaluated_at"
            ).alias(
                "earliest_evaluation"
            )
        )
        .first()["earliest_evaluation"]
    )

    latest_evaluation = (
        dataframe
        .agg(
            F.max(
                "evaluated_at"
            ).alias(
                "latest_evaluation"
            )
        )
        .first()["latest_evaluation"]
    )

    print()
    print("Table name          :", table_name)
    print("Record count        :", table_row_count)
    print("Run IDs             :", table_run_ids)
    print("Earliest evaluation :", earliest_evaluation)
    print("Latest evaluation   :", latest_evaluation)

print()
print("================================================")
print("FINAL GOLD ACCEPTANCE STATUS: PASSED")
print("================================================")

### New till here